# MuseTalk coarse lip-sync en Colab

Executa les cel·les en ordre en un runtime nou amb GPU T4. Si una cel·la d'instal·lació o descàrrega falla, atura't i copia l'error complet.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content
import shutil
from pathlib import Path
repo_dir = Path('/content/lipsync-pipeline')
if repo_dir.exists():
    shutil.rmtree(repo_dir)
!git clone --recurse-submodules https://github.com/arnaumartin10/SmartDub.git /content/lipsync-pipeline
%cd /content/lipsync-pipeline
!git submodule update --init --recursive
print('Repository cloned with submodules.')

/content
Cloning into '/content/lipsync-pipeline'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 114 (delta 49), reused 108 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 60.44 KiB | 12.09 MiB/s, done.
Resolving deltas: 100% (49/49), done.
Submodule 'third_party/MuseTalk' (https://github.com/TMElyralab/MuseTalk.git) registered for path 'third_party/MuseTalk'
Cloning into '/content/lipsync-pipeline/third_party/MuseTalk'...
remote: Enumerating objects: 534, done.        
remote: Total 534 (delta 0), reused 0 (delta 0), pack-reused 534 (from 1)        
Receiving objects: 100% (534/534), 25.75 MiB | 31.81 MiB/s, done.
Resolving deltas: 100% (214/214), done.
Submodule path 'third_party/MuseTalk': checked out '0a89dec45a0192b824e3cf4daf96c239440c5ed8'
/content/lipsync-pipeline
Repository cloned with submodules.


In [3]:
!nvidia-smi
!ffmpeg -version | head -n 2
import torch
print(f'Torch preinstalled: {torch.__version__}; CUDA: {torch.version.cuda}; available: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'CUDA is unavailable. Stop and copy the complete error.'

Tue Sep  8 14:53:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Instal·lació

Aquesta cel·la conserva el Torch/CUDA preinstal·lat per Colab. No instal·la `requirements.txt` complet perquè això podria substituir el runtime CUDA. No instal·la `opencv-python`: només s'utilitza `opencv-contrib-python`, compatible amb NumPy 2 i MediaPipe.

In [5]:
 %cd /content/lipsync-pipeline
import importlib.metadata as md
import subprocess, sys

def pip_install(*packages, no_deps=False):
    command = [sys.executable, '-m', 'pip', 'install', '--upgrade', '-q']
    if no_deps: command.append('--no-deps')
    subprocess.check_call(command + list(packages))

pip_install('diffusers==0.30.2', 'accelerate==0.28.0', 'transformers==4.48.3', 'tokenizers==0.21.4', 'huggingface_hub==0.36.2', 'whisperx==3.8.6', 'faster-whisper==1.2.0', 'ctranslate2==4.8.2', 'g2p_en==2.1.0', 'mediapipe==0.10.35', 'scenedetect==0.6.7', 'soundfile==0.12.1', 'librosa==0.10.2.post1', 'einops==0.8.1', 'omegaconf', 'ffmpeg-python', 'opencv-contrib-python==5.0.0.93', no_deps=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'opencv-python', 'opencv-contrib-python'], check=False, stdout=subprocess.DEVNULL)
pip_install('opencv-contrib-python==5.0.0.93', no_deps=True)
pip_install('onnxruntime>=1.20,<2', 'pyannote-audio==4.0.0', 'torchcodec==0.7.0', 'av', 'pandas', 'scipy', 'tqdm', 'pyyaml', 'sentencepiece', 'inflect==7.5.0', 'distance==0.1.3')

import numpy as np, cv2, mediapipe, scenedetect, onnxruntime, faster_whisper, whisperx, g2p_en
print(f'Torch: {torch.__version__}; CUDA: {torch.version.cuda}; available: {torch.cuda.is_available()}')
print('NumPy:', np.__version__)
print('OpenCV:', cv2.__version__)
print('MediaPipe:', mediapipe.__version__)
print('PySceneDetect:', scenedetect.__version__)
print('WhisperX:', md.version('whisperx'))
print('Tokenizers:', md.version('tokenizers'))
print('CTranslate2:', md.version('ctranslate2'))
print('ONNX Runtime:', md.version('onnxruntime'))
assert torch.cuda.is_available()
assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0)
assert tuple(int(x) for x in cv2.__version__.split('.')[:2]) >= (5, 0)
assert mediapipe.__version__ == '0.10.35'
assert md.version('whisperx') == '3.8.6'
assert md.version('tokenizers') == '0.21.4'
assert md.version('ctranslate2') == '4.8.2'
assert md.version('onnxruntime') == '1.29.0'
print('All dependency imports and version checks passed.')

/content/lipsync-pipeline
Torch: 2.11.0+cu128; CUDA: 12.8; available: True
NumPy: 2.1.3
OpenCV: 5.0.0
MediaPipe: 0.10.35
PySceneDetect: 0.6.7
WhisperX: 3.8.6
Tokenizers: 0.21.4
CTranslate2: 4.8.2
ONNX Runtime: 1.29.0
All dependency imports and version checks passed.


In [6]:
%cd /content/lipsync-pipeline
import subprocess
import sys
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '-q', '--no-deps', 'huggingface_hub==0.36.2', 'gdown'])
for directory in ('models/musetalkV15', 'models/sd-vae', 'models/whisper'):
    Path(directory).mkdir(parents=True, exist_ok=True)

# Modern MediaPipe Tasks needs this face-landmarker model; legacy Solutions does not.
face_model = Path('models/face_landmarker.task')
if not face_model.is_file():
    subprocess.check_call([
        'wget', '-q', '--show-progress', '-O', str(face_model),
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task',
    ])

def hf_download(repo, local_dir, *patterns):
    subprocess.check_call([
        'hf', 'download', repo, '--local-dir', local_dir,
        '--include', *patterns,
    ])

hf_download('TMElyralab/MuseTalk', 'models', 'musetalkV15/musetalk.json', 'musetalkV15/unet.pth')
hf_download('stabilityai/sd-vae-ft-mse', 'models/sd-vae', 'config.json', 'diffusion_pytorch_model.bin')
hf_download('openai/whisper-tiny', 'models/whisper', 'config.json', 'pytorch_model.bin', 'preprocessor_config.json')

required = [
    Path('models/face_landmarker.task'),
    Path('models/musetalkV15/musetalk.json'),
    Path('models/musetalkV15/unet.pth'),
    Path('models/sd-vae/config.json'),
    Path('models/sd-vae/diffusion_pytorch_model.bin'),
    Path('models/whisper/config.json'),
    Path('models/whisper/pytorch_model.bin'),
    Path('models/whisper/preprocessor_config.json'),
]
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise RuntimeError(f'Checkpoint download failed; missing or empty files: {missing}')
print('Download completed. Checkpoint files:')
for path in required:
    print(f'  {path}: {path.stat().st_size} bytes')

/content/lipsync-pipeline
Download completed. Checkpoint files:
  models/face_landmarker.task: 3758596 bytes
  models/musetalkV15/musetalk.json: 748 bytes
  models/musetalkV15/unet.pth: 3400074924 bytes
  models/sd-vae/config.json: 547 bytes
  models/sd-vae/diffusion_pytorch_model.bin: 334707217 bytes
  models/whisper/config.json: 1983 bytes
  models/whisper/pytorch_model.bin: 151095027 bytes
  models/whisper/preprocessor_config.json: 184990 bytes


In [24]:
%cd /content/lipsync-pipeline
from pathlib import Path

DRIVE_INPUT = Path('/content/lipsync-pipeline/data/inputs')
Path('data/inputs').mkdir(parents=True, exist_ok=True)
required_inputs = ['sample.mp4', 'sample_dub.wav', 'sample_dub.txt']
missing = [name for name in required_inputs if not (DRIVE_INPUT / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing Drive inputs in {DRIVE_INPUT}: {missing}')
for name in required_inputs:
    target = Path('data/inputs') / name
    target.write_bytes((DRIVE_INPUT / name).read_bytes())
print('Input files copied:')
for name in required_inputs:
    path = Path('data/inputs') / name
    print(f'  {path}: {path.stat().st_size} bytes')
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,nb_frames -of default=noprint_wrappers=1 data/inputs/sample.mp4
!ffprobe -v error -select_streams a:0 -show_entries stream=sample_rate,channels,duration -of default=noprint_wrappers=1 data/inputs/sample_dub.wav

/content/lipsync-pipeline
Input files copied:
  data/inputs/sample.mp4: 17282224 bytes
  data/inputs/sample_dub.wav: 431524 bytes
  data/inputs/sample_dub.txt: 248 bytes
width=1080
height=1920
r_frame_rate=30/1
nb_frames=404
sample_rate=16000
channels=1
duration=13.482688


In [29]:
%cd /content/lipsync-pipeline
from pathlib import Path
from src.generation.coarse_lipsync import CoarseLipSyncGenerator

assert Path('third_party/MuseTalk').is_dir()
assert Path('models/face_landmarker.task').is_file()
assert Path('models/musetalkV15/unet.pth').is_file()
assert Path('models/sd-vae/config.json').is_file()
assert Path('models/whisper/config.json').is_file()
print('Repository, checkpoints, MediaPipe model, and wrapper paths are ready.')

/content/lipsync-pipeline
Repository, checkpoints, MediaPipe model, and wrapper paths are ready.


In [ ]:
%cd /content/lipsync-pipeline
!git checkout scripts/generate_demo.py src/preprocessing/forced_alignment.py src/generation/coarse_lipsync.py

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '-q', 'accelerate'])

import nltk
for r in ('averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'cmudict'):
    nltk.download(r, quiet=True)
print('NLTK resources ready.')

from pathlib import Path

cl_path = Path('src/generation/coarse_lipsync.py')
if cl_path.is_file():
    cl_text = cl_path.read_text(encoding='utf-8')
    cl_text = cl_text.replace('vae_type=str(vae_dir)', 'vae_type=vae_dir.name')
    cl_path.write_text(cl_text, encoding='utf-8')
    print('Patched vae_type in coarse_lipsync.py')

patch = '''
import torch
_orig_torch_load = getattr(torch, "_orig_torch_load", torch.load)
torch._orig_torch_load = _orig_torch_load
def _always_safe_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _orig_torch_load(*args, **kwargs)
torch.load = _always_safe_load

try:
    import accelerate.utils.memory
    if not hasattr(accelerate.utils.memory, "clear_device_cache"):
        def clear_device_cache(*args, **kwargs):
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        accelerate.utils.memory.clear_device_cache = clear_device_cache
except Exception:
    pass
'''

for filepath in ('scripts/generate_demo.py', 'src/preprocessing/forced_alignment.py', 'src/generation/coarse_lipsync.py'):
    p = Path(filepath)
    content = p.read_text(encoding='utf-8')
    if 'from __future__ import annotations' in content:
        content = content.replace('from __future__ import annotations', 'from __future__ import annotations\n' + patch, 1)
    else:
        content = patch + '\n' + content
    p.write_text(content, encoding='utf-8')
    print(f'Patched {filepath}')

print('Fix applied successfully!')


In [35]:
%cd /content/lipsync-pipeline
!python scripts/generate_demo.py \
  --video data/inputs/sample.mp4 \
  --audio data/inputs/sample_dub.wav \
  --transcript data/inputs/sample_dub.txt \
  --checkpoint-dir models \
  --output data/outputs/musetalk_coarse.mp4

/content/lipsync-pipeline
INFO | Probed video: 404 frames @ 30.00 fps
INFO | Detecting scenes...
WARNING | ContentDetector found no cuts in sample.mp4 (threshold=27.0). Returning entire video as a single scene.
INFO | NumExpr defaulting to 2 threads.
W0000 00:00:1788880456.157151    7435 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1788880456.167064    7438 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788880456.187571    7438 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
INFO | MediaPipe Tasks scene frames 0–403: 404/404 faces detected, 0 missed
WARNING | /usr/local/lib/python3.13/dist-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed c

In [33]:
%cd /content/lipsync-pipeline
from pathlib import Path
output = Path('/content/lipsync-pipeline/data/outputs/musetalk_coarse.mp4')
assert output.is_file() and output.stat().st_size > 0, 'MuseTalk output was not created.'
!ffprobe -v error -show_entries format=duration -show_entries stream=width,height,r_frame_rate,nb_frames -of default=noprint_wrappers=1 data/outputs/musetalk_coarse.mp4
drive_output = Path('/content//SmartDub/data/outputs')
drive_output.mkdir(parents=True, exist_ok=True)
(drive_output / output.name).write_bytes(output.read_bytes())
print(f'Copied result to {drive_output / output.name}')

/content/lipsync-pipeline


AssertionError: MuseTalk output was not created.